In [3]:
"hello, i m building an ai project"

'hello, i m building an ai project'

In [4]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 38.5 MB/s eta 0:00:00


In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
data = pd.read_csv(url)

data.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1


In [6]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def get_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return pd.Series({
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol)
    })

features = data["smiles"].apply(get_features)
data_final = pd.concat([data, features], axis=1)
data_final.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles,MolWt,LogP,NumHDonors,NumHAcceptors,TPSA,NumRotatableBonds,RingCount
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,457.432,-3.10802,7.0,12.0,202.32,7.0,3.0
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2,201.225,2.84032,1.0,2.0,42.24,2.0,2.0
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O),152.237,2.87800,0.0,1.0,17.07,4.0,0.0
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,278.354,6.29940,0.0,0.0,0.00,0.0,5.0
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1,84.143,1.74810,0.0,1.0,0.00,0.0,1.0


In [7]:
from sklearn.model_selection import train_test_split

feature_cols = ["MolWt", "LogP", "NumHDonors", "NumHAcceptors", "TPSA", "NumRotatableBonds", "RingCount"]

X = data_final[feature_cols]
y = data_final["measured log solubility in mols per litre"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training molecules:", X_train.shape[0])
print("Testing molecules:", X_test.shape[0])

Training molecules: 902
Testing molecules: 226


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("R² score:", r2_score(y_test, predictions))
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, predictions))
print("RMSE:", rmse)

R² score: 0.7580799586498999
RMSE: 1.0693495666199677


In [10]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

print("Random Forest R² score:", r2_score(y_test, rf_predictions))
print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_test, rf_predictions)))

Random Forest R² score: 0.8609747333125335
Random Forest RMSE: 0.8106448764441023


In [11]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)
print(importances)

LogP                 0.816315
MolWt                0.100521
TPSA                 0.042982
RingCount            0.011818
NumRotatableBonds    0.010945
NumHAcceptors        0.010798
NumHDonors           0.006621
dtype: float64


In [12]:
def predict_solubility(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid SMILES string — please check it."

    feats = pd.DataFrame([{
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol)
    }])

    pred = rf_model.predict(feats)[0]
    return round(pred, 3)

# Try it on caffeine
print("Caffeine solubility (predicted):", predict_solubility("CN1C=NC2=C1C(=O)N(C(=O)N2C)C"))

# Try it on aspirin
print("Aspirin solubility (predicted):", predict_solubility("CC(=O)OC1=CC=CC=C1C(=O)O"))

Caffeine solubility (predicted): -1.076
Aspirin solubility (predicted): -1.898


In [13]:
print("Sodium atom:", predict_solubility("[Na]"))
print("Sodium ion:", predict_solubility("[Na+]"))

Sodium atom: 0.857
Sodium ion: 0.733


In [14]:
print("Sodium chloride:", predict_solubility("[Na+].[Cl-]"))

Sodium chloride: 0.698


In [17]:
!pip install pubchempy

In [18]:
import pubchempy as pcp

def name_to_smiles(name):
    try:
        compound = pcp.get_compounds(name, 'name')
        if compound:
            return compound[0].isomeric_smiles
        else:
            return None
    except Exception as e:
        return None

def predict_by_name(name):
    smiles = name_to_smiles(name)
    if smiles is None:
        return f"Could not find a molecule named '{name}'. Try checking the spelling, or use a SMILES string instead."

    result = predict_solubility(smiles)
    return f"{name} (SMILES: {smiles}) → Predicted solubility: {result}"

print(predict_by_name("aspirin"))

/tmp/ipykernel_824/714012850.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compound[0].isomeric_smiles


aspirin (SMILES: CC(=O)OC1=CC=CC=C1C(=O)O) → Predicted solubility: -1.898


In [19]:
def name_to_smiles(name):
    try:
        compound = pcp.get_compounds(name, 'name')
        if compound:
            return compound[0].smiles
        else:
            return None
    except Exception as e:
        return None

In [20]:
print(predict_by_name("aspirin"))
print(predict_by_name("caffeine"))
print(predict_by_name("ethanol"))

aspirin (SMILES: CC(=O)OC1=CC=CC=C1C(=O)O) → Predicted solubility: -1.898
caffeine (SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C) → Predicted solubility: -1.076
ethanol (SMILES: CCO) → Predicted solubility: 0.98


In [21]:
!pip install gradio

In [22]:
import gradio as gr

def gradio_predict(name):
    smiles = name_to_smiles(name)
    if smiles is None:
        return f"Could not find a molecule named '{name}'. Try checking spelling."
    result = predict_solubility(smiles)
    return f"SMILES: {smiles}\nPredicted log solubility: {result}"

interface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(label="Enter a molecule name (e.g. aspirin, caffeine, ethanol)"),
    outputs=gr.Textbox(label="Result"),
    title="Molecule Solubility Predictor",
    description="Type a molecule name to predict its water solubility using a Random Forest model trained on RDKit chemistry features."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e0d2052a96409306f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
from rdkit.Chem import Draw

def interpret_solubility(pred, logp, tpsa):
    if pred > -1:
        level = "highly soluble in water"
    elif pred > -3:
        level = "moderately soluble in water"
    else:
        level = "poorly soluble in water"

    if logp > 3:
        reason = "it is quite fat-loving (high LogP), so it resists mixing with water"
    elif logp < 0:
        reason = "it is water-loving (low/negative LogP), which helps it dissolve"
    else:
        reason = "it has balanced fat/water character (moderate LogP)"

    return f"This molecule is predicted to be {level}. Main reason: {reason}."

def gradio_predict(name):
    smiles = name_to_smiles(name)
    if smiles is None:
        return None, f"Could not find a molecule named '{name}'. Try checking spelling."

    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, size=(300, 300))

    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    pred = predict_solubility(smiles)

    explanation = interpret_solubility(pred, logp, tpsa)

    text_result = (
        f"SMILES: {smiles}\n"
        f"Predicted log solubility: {pred}\n"
        f"(Negative = less soluble; closer to 0 or positive = more soluble)\n\n"
        f"{explanation}"
    )

    return img, text_result

interface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(label="Enter a molecule name (e.g. aspirin, caffeine, ethanol)"),
    outputs=[gr.Image(label="Molecule Structure"), gr.Textbox(label="Result")],
    title="Molecule Solubility Predictor",
    description="Type a molecule name to see its structure and predicted water solubility."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36bb894c1909c65555.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
